In [1]:
# Import libraries
import pandas as pd

In [6]:
# Read in positive samples (ANOMALIES)
df_cloudflare_anomalies = pd.read_csv("./data/cloudflare_anomalies.csv")
df_ttnet_anomalies = pd.read_csv("./data/ttnet_anomalies.csv")
df_google_anomalies = pd.read_csv("./data/google_anomalies.csv")

df_positive_samples = pd.concat([df_cloudflare_anomalies, df_ttnet_anomalies, df_google_anomalies], ignore_index=True)
df_positive_samples["label"] = 1
df_positive_samples.head()

,source_node,destination_node,timestamp,edge_idx,label,node_features,edge_features
0,6730,6830,1719514200,0,1,0.0,1.0
1,6830,3356,1719514200,1,1,0.0,1.0
2,3356,263608,1719514200,2,1,0.0,1.0
3,6730,6830,1719514200,3,1,0.0,1.0
4,6830,174,1719514200,4,1,0.0,1.0


In [7]:
# Read in negative samples 
df_cloudflare_before = pd.read_csv("./data/cloudflare_day_before.csv")
df_cloudflare_after = pd.read_csv("./data/cloudflare_day_after.csv")
df_ttnet_before = pd.read_csv("./data/ttnet_day_before.csv")
df_ttnet_after = pd.read_csv("./data/ttnet_day_after.csv")
df_google_before = pd.read_csv("./data/google_day_before.csv")
df_google_after = pd.read_csv("./data/google_day_after.csv")

df_negative_samples = pd.concat([df_cloudflare_before, df_cloudflare_after, df_ttnet_before, df_ttnet_after, df_google_before, df_google_after], ignore_index=True)
df_negative_samples["label"] = 0
df_negative_samples.head()

,source_node,destination_node,timestamp,edge_idx,label,node_features,edge_features
0,15547,174,1719427800,0,0,0.0,1.0
1,174,3491,1719427800,1,0,0.0,1.0
2,3491,12389,1719427800,2,0,0.0,1.0
3,12389,33991,1719427800,3,0,0.0,1.0
4,33991,209372,1719427800,4,0,0.0,1.0


In [8]:
df = pd.concat([df_positive_samples, df_negative_samples], ignore_index=True)
df = df.sort_values("timestamp")  # important for TGN!

df.to_csv("./data/tgn_events.csv", index=False)

In [9]:
from pathlib import Path

def split_tgn_csv(
    input_csv="./tgn_events.csv",
    output_dir="./data/bgp_split",
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15
):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"

    # Load and sort by timestamp
    df = pd.read_csv(input_csv)
    df = df.sort_values("timestamp").reset_index(drop=True)

    # Split indices
    total = len(df)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    df_train = df.iloc[:train_end]
    df_val = df.iloc[train_end:val_end]
    df_test = df.iloc[val_end:]

    # Create output dir
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    df_train.to_csv(f"{output_dir}/train.csv", index=False)
    df_val.to_csv(f"{output_dir}/val.csv", index=False)
    df_test.to_csv(f"{output_dir}/test.csv", index=False)

    print(f"✅ Saved {len(df_train)} train, {len(df_val)} val, {len(df_test)} test samples to {output_dir}/")

# Example usage
split_tgn_csv("./data/tgn_events.csv", output_dir="./data/bgp_split")


✅ Saved 635939 train, 136272 val, 136274 test samples to ./data/bgp_split/
